# In-context inference of grid transformations — k = 2, `openai/gpt-6-luna` via OpenRouter

The model sees 3 input/output pairs of 6×6 digit grids, all produced by the same hidden transformation, and must
apply it to a new grid. The transformations are the 6 ordered pairs of distinct operations: AB, AC, BA, BC, CA, CB
(A = shift every row right by one with wrap-around, B = rotate 90° clockwise, C = mirror left–right; left to right).
The prompt lists the three operations but never says which ones are used.

**Two experiments on the same tasks.** They differ only in the final instruction:
* `answer`: give the output grid;
* `ops`: first state the transformation as a sequence of the operations (`<ops>…</ops>`), then give the grid.

**Task construction.** The demos are checked so that exactly one of the 288 transformations fits them. The query
grid has 288 distinct images, so every answer identifies the transformation it corresponds to (or none).

**Before running.**
1. Create an OpenRouter API key (openrouter.ai → Keys; set a credit limit).
2. In Colab, open the key icon (Secrets) on the left and add a secret named `OPENROUTER_API_KEY` with the key as its
   value; allow this notebook to access it. Never paste the key into a cell.

**Order.** Cell 4 runs a dry run (no API calls, no cost). Cell 5 runs a 4-call pilot: check the responses and
the cost. Cell 6 runs everything (resumable: calls already made are skipped). Cell 7 scores and downloads the
results. A CPU runtime is enough.


In [1]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


Repository: /content/Decompositionality | commit: 44062dfb77fd5f744d8f3d04eaabd91dab15c6fd


In [2]:
# --- 3. Configuration ---------------------------------------------------------------------------------------------
MODEL = "openai/gpt-6-luna"
N_PER_WORD = 20                  # tasks per transformation (6 transformations -> 120 tasks, 240 calls for 2 experiments)
N_DEMOS = 3
TASK_SEED = 0
EXPERIMENTS = ["answer", "ops"]
REASONING = None                 # None = the model's default; e.g. {"effort": "high"} or {"max_tokens": 8000}
MAX_TOKENS = None                # cap on output tokens per call (None = provider default)
WORKERS = 4                      # parallel API calls
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if IN_COLAB and not os.environ.get("OPENROUTER_API_KEY"):
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")      # never printed
print("OpenRouter key present:", bool(os.environ.get("OPENROUTER_API_KEY")))
import json, pathlib
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 300); pd.set_option("display.width", 220)
from decompositionality import icl_api as IA
_tag = "default" if not REASONING else "_".join(f"{k}{v}" for k, v in REASONING.items())
OUT_DIR = pathlib.Path(RESULTS_ROOT) / "icl_api" / f"k2_{MODEL.replace('/', '_')}_demos{N_DEMOS}_seed{TASK_SEED}_{_tag}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSONL = OUT_DIR / "responses.jsonl"
print("results ->", OUT_DIR)


Mounted at /content/drive
OpenRouter key present: True
results -> /content/drive/MyDrive/decompositionality_results/icl_api/k2_openai_gpt-6-luna_demos3_seed0_default


In [3]:
# --- 4. Dry run: build the tasks and show one prompt per experiment (no API calls) ----------------------------------
tasks = IA.make_tasks(IA.K2_WORDS, n_per_word=N_PER_WORD, n_demos=N_DEMOS, seed=TASK_SEED)
(OUT_DIR / "tasks.json").write_text(json.dumps(tasks))
print(len(tasks), "tasks:", {w: sum(t["word"] == w for t in tasks) for w in IA.K2_WORDS})
for e in EXPERIMENTS:
    print(f"\n==================== experiment '{e}' (task {tasks[0]['task_id']}) ====================")
    print(IA.build_prompt(tasks[0], e))
print("\nExpected answer for", tasks[0]["task_id"], ":")
print(IA._fmt(tasks[0]["expected"]))


120 tasks: {'AB': 20, 'AC': 20, 'BA': 20, 'BC': 20, 'CA': 20, 'CB': 20}

==================== experiment 'answer' (task AB-000) ====================
Each example below shows a 6x6 grid of digits (Input) and the grid obtained by applying the same hidden transformation to it (Output). The transformation is the same for every example. The hidden transformation is a sequence of one or more of these operations:
(A) shift every row one cell to the right, wrapping around;
(B) rotate the grid 90 degrees clockwise;
(C) mirror the grid left-to-right.

Example 1
Input:
7 7 9 4 3 4
8 9 0 4 0 8
2 2 3 8 0 8
7 9 9 0 9 8
7 4 7 5 0 6
4 7 4 9 8 6
Output:
6 6 8 8 8 4
4 7 7 2 8 7
7 4 9 2 9 7
4 7 9 3 0 9
9 5 0 8 4 4
8 0 9 0 0 3

Example 2
Input:
9 5 7 7 8 4
6 3 1 4 9 0
0 5 0 7 3 6
2 0 4 8 6 8
1 7 6 1 4 3
7 2 4 9 5 5
Output:
5 3 8 6 0 4
7 1 2 0 6 9
2 7 0 5 3 5
4 6 4 0 1 7
9 1 8 7 4 7
5 4 6 3 9 8

Example 3
Input:
4 9 7 9 4 3
4 6 6 8 9 0
3 2 3 3 1 3
3 8 6 8 4 9
5 5 2 9 6 9
9 0 0 0 7 3
Output:
3 9 9 3 0 3
9 5

In [4]:
# --- 5. Pilot: 4 calls, then inspect the raw responses and the cost ------------------------------------------------
IA.run(tasks, OUT_JSONL, MODEL, experiments=EXPERIMENTS, reasoning=REASONING, max_tokens=MAX_TOKENS, workers=WORKERS,
       limit=4)
for line in OUT_JSONL.read_text().splitlines()[-4:]:
    r = json.loads(line)
    print(f"\n----- {r['task_id']} / {r['experiment']} | model {r['model']} | {r['seconds']} s | error: {r['error']}")
    print((r["content"] or "")[:1500])
    print("usage:", r["usage"])
display(IA.score(tasks, OUT_JSONL))


0 calls already done; 4 to run (model openai/gpt-6-luna)
  4/4 calls finished (4 ok)

----- AB-001 / ops | model openai/gpt-6-luna | 13.8 s | error: None
<ops>A, then B</ops>
<answer>
3 8 7 0 3 5
8 6 4 9 1 5
1 1 5 4 9 2
7 1 2 4 0 9
9 4 4 9 4 3
2 0 4 4 5 8
</answer>
usage: {'prompt_tokens': 683, 'completion_tokens': 1251, 'total_tokens': 1934, 'cost': 0.0006938, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.0006938, 'upstream_inference_prompt_cost': 6.83e-05, 'upstream_inference_completions_cost': 0.0006255}, 'completion_tokens_details': {'reasoning_tokens': 1158, 'image_tokens': 0, 'audio_tokens': 0}}

----- AB-000 / ops | model openai/gpt-6-luna | 16.1 s | error: None
<ops>A, then B</ops>
<answer>
8 0 0 7 6 1
8 9 9 7 5 5
2 5 6 3 0 3
2 4 9 0 6 2
8 9 1 4 8 9
8 2 7 3 9 8
</answer>
usage: {'prompt_tokens': 683, 'completion_tokens': 1622, 'total_tokens': 2305, 'co

,task_id,word,experiment,parsed,correct,applied,ops_stated,ops_equivalent,ops_literal,ops_consistent_with_answer,prompt_tokens,completion_tokens,reasoning_tokens,cost_usd
0,AB-001,AB,ops,True,True,AB,AB,True,True,True,683,1251,1158,0.000694
1,AB-000,AB,ops,True,True,AB,AB,True,True,True,683,1622,1529,0.000879
2,AB-000,AB,answer,True,True,AB,NaN,NaN,NaN,NaN,662,2041,1957,0.001087
3,AB-001,AB,answer,True,True,AB,NaN,NaN,NaN,NaN,662,2904,2820,0.001518


In [5]:
# --- 6. Full run (resumable: calls already in responses.jsonl are skipped) -------------------------------------------
IA.run(tasks, OUT_JSONL, MODEL, experiments=EXPERIMENTS, reasoning=REASONING, max_tokens=MAX_TOKENS, workers=WORKERS)


4 calls already done; 236 to run (model openai/gpt-6-luna)
  10/236 calls finished (10 ok)
  20/236 calls finished (20 ok)
  30/236 calls finished (30 ok)
  40/236 calls finished (40 ok)
  50/236 calls finished (50 ok)
  60/236 calls finished (60 ok)
  70/236 calls finished (70 ok)
  80/236 calls finished (80 ok)
  90/236 calls finished (90 ok)
  100/236 calls finished (100 ok)
  110/236 calls finished (110 ok)
  120/236 calls finished (120 ok)
  130/236 calls finished (130 ok)
  140/236 calls finished (140 ok)
  150/236 calls finished (150 ok)
  160/236 calls finished (160 ok)
  170/236 calls finished (170 ok)
  180/236 calls finished (180 ok)
  190/236 calls finished (190 ok)
  200/236 calls finished (200 ok)
  210/236 calls finished (210 ok)
  220/236 calls finished (220 ok)
  230/236 calls finished (230 ok)
  236/236 calls finished (236 ok)


236

In [6]:
# --- 7. Scores and summary --------------------------------------------------------------------------------------------
df = IA.score(tasks, OUT_JSONL)
S = IA.summarize(df)
for name, tab in S.items():
    print(f"\n### {name}")
    display(tab)
df.to_csv(OUT_DIR / "scores.csv", index=False)
for name, tab in S.items():
    tab.to_csv(OUT_DIR / f"summary_{name}.csv", index=False)
print("saved to", OUT_DIR)
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    files.download(str(OUT_DIR) + ".zip")



### accuracy


experiment,word,answer,ops
0,AB,0.80,0.60
1,AC,0.95,0.80
2,BA,0.80,0.85
3,BC,1.00,1.00
4,CA,0.65,0.75
5,CB,0.85,0.95



### n_calls


,experiment,calls
0,answer,120
1,ops,120



### wrong_answers


,experiment,word,applied,count
1,answer,AB,not a legal transform,3
0,answer,AB,B,1
2,answer,AC,not a legal transform,1
4,answer,BA,not a legal transform,3
3,answer,BA,CABA,1
5,answer,CA,C,5
6,answer,CA,not a legal transform,2
8,answer,CB,not a legal transform,2
7,answer,CB,B,1
11,ops,AB,not a legal transform,6


... (5 outputs left out)

,prompt_tokens,completion_tokens,reasoning_tokens,cost_usd
total,161400.0,286848.0,265394.0,0.159564


saved to /content/drive/MyDrive/decompositionality_results/icl_api/k2_openai_gpt-6-luna_demos3_seed0_default


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>